In [3]:
# Import libraries

from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

import joblib

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

In [4]:
# Define project paths

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

FEATURES_DIR = PROJECT_ROOT / "data" / "03_features"
PREDICTIONS_DIR = PROJECT_ROOT / "data" / "04_predictions"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FEATURES_DIR:", FEATURES_DIR)
print("PREDICTIONS_DIR:", PREDICTIONS_DIR)
print("MODELS_DIR:", MODELS_DIR)

PROJECT_ROOT: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform
FEATURES_DIR: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\03_features
PREDICTIONS_DIR: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\04_predictions
MODELS_DIR: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\models


In [5]:
# Load claim feature data from Notebook 2

input_path = FEATURES_DIR / "claim_features.csv"

claims_df = pd.read_csv(input_path, low_memory=False)

print("Shape:", claims_df.shape)
display(claims_df.head())

Shape: (857563, 14)


,member_id,claim_id,claim_start_date,claim_end_date,claim_payment_amount,provider_id,primary_diagnosis_code,claim_type,claim_duration_days,high_cost_claim,claim_payment_amount_clipped,claim_payment_log,has_provider_id,has_diagnosis_code
0,00013D2EFD8E45D1,196661176988405,2010-03-12,2010-03-13,4000.0,2600GD,4580,inpatient,2.0,1,4000.0,8.294300,1,1
1,00016F745862898F,196201177000368,2009-04-12,2009-04-18,26000.0,3900MB,7866,inpatient,7.0,1,26000.0,10.165890,1,1
2,00016F745862898F,196661177015632,2009-08-31,2009-09-02,5000.0,3900HM,6186,inpatient,3.0,1,5000.0,8.517393,1,1
3,00016F745862898F,196091176981058,2009-09-17,2009-09-20,5000.0,3913XU,29590,inpatient,4.0,1,5000.0,8.517393,1,1
4,00016F745862898F,196261176983265,2010-06-26,2010-07-01,16000.0,3900MB,5849,inpatient,6.0,1,16000.0,9.680406,1,1


In [6]:
# Inspect columns and target distribution

print("Columns:")
print(claims_df.columns.tolist())

print("\nTarget distribution:")
display(claims_df["high_cost_claim"].value_counts().to_frame("count"))
display(claims_df["high_cost_claim"].value_counts(normalize=True).to_frame("rate"))

Columns:
['member_id', 'claim_id', 'claim_start_date', 'claim_end_date', 'claim_payment_amount', 'provider_id', 'primary_diagnosis_code', 'claim_type', 'claim_duration_days', 'high_cost_claim', 'claim_payment_amount_clipped', 'claim_payment_log', 'has_provider_id', 'has_diagnosis_code']

Target distribution:


,count
high_cost_claim,
0,769740
1,87823


,rate
high_cost_claim,
0,0.89759
1,0.10241


In [7]:
# Select modeling columns
# Avoid leakage: do not use raw claim_payment_amount or claim_payment_log,
# because high_cost_claim was created directly from payment amount.

target_col = "high_cost_claim"

feature_cols = [
    "claim_type",
    "claim_duration_days",
    "has_provider_id",
    "has_diagnosis_code",
]

model_df = claims_df[feature_cols + [target_col]].copy()

print("Model dataset shape:", model_df.shape)
display(model_df.head())

Model dataset shape: (857563, 5)


,claim_type,claim_duration_days,has_provider_id,has_diagnosis_code,high_cost_claim
0,inpatient,2.0,1,1,1
1,inpatient,7.0,1,1,1
2,inpatient,3.0,1,1,1
3,inpatient,4.0,1,1,1
4,inpatient,6.0,1,1,1


In [8]:
# Check missing values in modeling data

missing_summary = model_df.isna().sum().reset_index()
missing_summary.columns = ["column", "missing_count"]
missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(model_df) * 100).round(2)

display(missing_summary)

,column,missing_count,missing_pct
0,claim_type,0,0.00
1,claim_duration_days,11321,1.32
2,has_provider_id,0,0.00
3,has_diagnosis_code,0,0.00
4,high_cost_claim,0,0.00


In [9]:
# Split features and target

X = model_df[feature_cols]
y = model_df[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Positive class rate:", y.mean().round(4))

X shape: (857563, 4)
y shape: (857563,)
Positive class rate: 0.1024


In [10]:
# Train/test split with stratification

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Train positive rate:", y_train.mean().round(4))
print("Test positive rate:", y_test.mean().round(4))

X_train: (686050, 4)
X_test: (171513, 4)
Train positive rate: 0.1024
Test positive rate: 0.1024


In [11]:
# Define numeric and categorical preprocessing

numeric_features = [
    "claim_duration_days",
    "has_provider_id",
    "has_diagnosis_code",
]

categorical_features = [
    "claim_type",
]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [12]:
# Create reusable evaluation function

def evaluate_classifier(model, X_train, y_train, X_test, y_test, model_name):
    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        train_proba = model.predict_proba(X_train)[:, 1]
        test_proba = model.predict_proba(X_test)[:, 1]
    else:
        train_proba = None
        test_proba = None

    results = {
        "model": model_name,
        "train_accuracy": accuracy_score(y_train, train_preds),
        "test_accuracy": accuracy_score(y_test, test_preds),
        "train_precision": precision_score(y_train, train_preds, zero_division=0),
        "test_precision": precision_score(y_test, test_preds, zero_division=0),
        "train_recall": recall_score(y_train, train_preds, zero_division=0),
        "test_recall": recall_score(y_test, test_preds, zero_division=0),
        "train_f1": f1_score(y_train, train_preds, zero_division=0),
        "test_f1": f1_score(y_test, test_preds, zero_division=0),
    }

    if train_proba is not None:
        results["train_roc_auc"] = roc_auc_score(y_train, train_proba)
        results["test_roc_auc"] = roc_auc_score(y_test, test_proba)

    return results

In [13]:
# Baseline model: Logistic Regression
# This gives a simple benchmark before using stronger models.

log_reg_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

log_reg_model.fit(X_train, y_train)

log_reg_results = evaluate_classifier(
    log_reg_model,
    X_train,
    y_train,
    X_test,
    y_test,
    "Logistic Regression Baseline",
)

log_reg_results

{'model': 'Logistic Regression Baseline',
 'train_accuracy': 0.9266059325122076,
 'test_accuracy': 0.9261047267554063,
 'train_precision': 0.6047948955525606,
 'test_precision': 0.6029727567476525,
 'train_recall': 0.8175723760995189,
 'test_recall': 0.8152576145744378,
 'train_f1': 0.6952685282690003,
 'test_f1': 0.6932274773684465,
 'train_roc_auc': 0.9069867847726059,
 'test_roc_auc': 0.905576701064772}

In [14]:
# Stronger model: Random Forest
# Keep it controlled to avoid unnecessary training time and overfitting.

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=150,
                max_depth=8,
                min_samples_leaf=20,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

rf_model.fit(X_train, y_train)

rf_results = evaluate_classifier(
    rf_model,
    X_train,
    y_train,
    X_test,
    y_test,
    "Random Forest",
)

rf_results

{'model': 'Random Forest',
 'train_accuracy': 0.937395233583558,
 'test_accuracy': 0.9370718254592947,
 'train_precision': 0.6577111439659952,
 'test_precision': 0.6568900009266981,
 'train_recall': 0.8104699820660992,
 'test_recall': 0.8071164247082265,
 'train_f1': 0.7261435658084344,
 'test_f1': 0.724295603749968,
 'train_roc_auc': 0.9070885755802863,
 'test_roc_auc': 0.9056158279950811}

In [15]:
# Compare model results

results_df = pd.DataFrame(
    [
        log_reg_results,
        rf_results,
    ]
)

metric_cols = [
    "model",
    "train_accuracy",
    "test_accuracy",
    "train_precision",
    "test_precision",
    "train_recall",
    "test_recall",
    "train_f1",
    "test_f1",
    "train_roc_auc",
    "test_roc_auc",
]

results_df = results_df[metric_cols]

display(results_df)

,model,train_accuracy,test_accuracy,train_precision,test_precision,train_recall,test_recall,train_f1,test_f1,train_roc_auc,test_roc_auc
0,Logistic Regression Baseline,0.926606,0.926105,0.604795,0.602973,0.817572,0.815258,0.695269,0.693227,0.906987,0.905577
1,Random Forest,0.937395,0.937072,0.657711,0.656890,0.810470,0.807116,0.726144,0.724296,0.907089,0.905616


In [16]:
# Check overfitting gap

results_df["roc_auc_gap"] = results_df["train_roc_auc"] - results_df["test_roc_auc"]
results_df["f1_gap"] = results_df["train_f1"] - results_df["test_f1"]

display(
    results_df[
        [
            "model",
            "train_roc_auc",
            "test_roc_auc",
            "roc_auc_gap",
            "train_f1",
            "test_f1",
            "f1_gap",
        ]
    ]
)

,model,train_roc_auc,test_roc_auc,roc_auc_gap,train_f1,test_f1,f1_gap
0,Logistic Regression Baseline,0.906987,0.905577,0.001410,0.695269,0.693227,0.002041
1,Random Forest,0.907089,0.905616,0.001473,0.726144,0.724296,0.001848


In [17]:
# Select best model by test ROC-AUC

best_model_name = results_df.sort_values("test_roc_auc", ascending=False).iloc[0]["model"]

if best_model_name == "Random Forest":
    best_model = rf_model
else:
    best_model = log_reg_model

print("Best model:", best_model_name)

Best model: Random Forest


In [19]:
# Detailed test-set evaluation for the best model

test_preds = best_model.predict(X_test)
test_proba = best_model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, test_preds, zero_division=0))

print("Confusion Matrix:")
print(confusion_matrix(y_test, test_preds))

print("Test ROC-AUC:", round(roc_auc_score(y_test, test_proba), 4))

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.95      0.96    153948
           1       0.66      0.81      0.72     17565

    accuracy                           0.94    171513
   macro avg       0.82      0.88      0.84    171513
weighted avg       0.94      0.94      0.94    171513

Confusion Matrix:
[[146543   7405]
 [  3388  14177]]
Test ROC-AUC: 0.9056


In [20]:
# Create prediction output for reporting

predictions_df = X_test.copy()
predictions_df["actual_high_cost_claim"] = y_test.values
predictions_df["predicted_high_cost_claim"] = test_preds
predictions_df["high_cost_claim_probability"] = test_proba

display(predictions_df.head())

,claim_type,claim_duration_days,has_provider_id,has_diagnosis_code,actual_high_cost_claim,predicted_high_cost_claim,high_cost_claim_probability
488679,outpatient,1.0,1,1,0,0,0.147173
328476,outpatient,1.0,1,1,0,0,0.147173
432658,outpatient,1.0,1,1,0,0,0.147173
177493,outpatient,1.0,1,1,0,0,0.147173
334799,outpatient,1.0,1,1,0,0,0.147173


In [21]:
# Save model results and predictions

model_results_output_path = REPORTS_DIR / "claim_model_results.csv"
predictions_output_path = PREDICTIONS_DIR / "claim_risk_predictions.csv"

results_df.to_csv(model_results_output_path, index=False)
predictions_df.to_csv(predictions_output_path, index=False)

print("Saved model results:", model_results_output_path)
print("Saved predictions:", predictions_output_path)

Saved model results: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\reports\claim_model_results.csv
Saved predictions: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\04_predictions\claim_risk_predictions.csv


In [22]:
# Save best model artifact

model_output_path = MODELS_DIR / "claim_risk_model.pkl"

joblib.dump(best_model, model_output_path)

print("Saved best model:", model_output_path)

Saved best model: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\models\claim_risk_model.pkl


In [23]:
# Reload model to confirm it works

loaded_model = joblib.load(model_output_path)

sample_predictions = loaded_model.predict(X_test.head())
sample_probabilities = loaded_model.predict_proba(X_test.head())[:, 1]

print("Sample predictions:", sample_predictions)
print("Sample probabilities:", sample_probabilities)

Sample predictions: [0 0 0 0 0]
Sample probabilities: [0.14717329 0.14717329 0.14717329 0.14717329 0.14717329]


In [25]:
# Write simple model card

model_card_path = REPORTS_DIR / "model_card.md"

try:
    results_table = results_df.to_markdown(index=False)
except ImportError:
    results_table = results_df.to_string(index=False)

model_card_text = f"""
# Claim Risk Model Card

## Model Purpose

This model predicts whether a CMS synthetic Medicare claim is likely to be high-cost.

## Dataset

The model uses CMS DE-SynPUF synthetic claims data prepared in Notebook 2.

## Target Variable

The target variable is `high_cost_claim`.

A claim is labeled high-cost if its payment amount is greater than or equal to the 90th percentile threshold created during claims EDA.

## Features Used

The model uses the following non-leakage features:

1. claim_type
2. claim_duration_days
3. has_provider_id
4. has_diagnosis_code

Payment amount fields were excluded from training because the target was created from payment amount.

## Best Model

Best model selected by test ROC-AUC:

{best_model_name}

## Model Results

{results_table}

## Risk and Limitations

1. This model uses synthetic Medicare claims data, not real protected health information.
2. The target is a proxy label based on claim payment amount.
3. Payment amount fields are excluded from training to avoid target leakage.
4. This model should be used for triage and prioritization, not automatic claim decisions.
5. Human review should remain part of any operational workflow.

## Intended Use

This model demonstrates claims risk modeling for benefits administration workflows.
It can support prioritization, analytics, and internal review queues.
"""

model_card_path.write_text(model_card_text.strip(), encoding="utf-8")

print("Saved model card:", model_card_path)

Saved model card: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\reports\model_card.md


In [26]:
# Final notebook summary

print("Notebook 3 complete.")
print("Best model:", best_model_name)
print("Rows used:", len(model_df))
print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nFiles created:")
print("1. reports/claim_model_results.csv")
print("2. data/04_predictions/claim_risk_predictions.csv")
print("3. models/claim_risk_model.pkl")
print("4. reports/model_card.md")

Notebook 3 complete.
Best model: Random Forest
Rows used: 857563
Train rows: 686050
Test rows: 171513

Files created:
1. reports/claim_model_results.csv
2. data/04_predictions/claim_risk_predictions.csv
3. models/claim_risk_model.pkl
4. reports/model_card.md
